# Inspection des Données Traitées (Architecture 'Model Collapse Fix')

Ce notebook charge les données brutes et applique le nouveau pipeline d'ingénierie des fonctionnalités pour visualiser :
1. Les nouvelles features de "Contexte Positionnel" (`dist_ema`, `rsi_norm`, etc.)
2. Les statistiques descriptives.
3. La présence d'outliers (avant et après RobustScaler).

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ajouter la racine du projet au path pour pouvoir importer 'app'
sys.path.append(os.path.abspath("../../.."))

from app.MLearning.preprocessing.CryptoDataProcessor import CryptoDataProcessor

In [2]:
# Initialiser le processeur
processor = CryptoDataProcessor(scaler_path="artifacts/scaler.pkl")

# Charger les données brutes
input_path = "../../../data/btc_15m_data_2018_to_2025.csv"
if not os.path.exists(input_path):
    # Fallback si exécuté depuis un autre dossier
    input_path = "data/btc_15m_data_2018_to_2025.csv"

print(f"Chargement de {input_path}...")
df = processor.load_data(input_path)

## 1. Génération des Features
Nous appliquons les étapes manuelles du pipeline pour inspecter le DataFrame intermédiaire.

In [3]:
# 1. Features Saisonnières
df = processor.add_Seasonal_features(df)

# 2. Indicateurs Techniques (Production Parity)
df = processor.calculate_technical_indicators(df)

# 3. Features Positionnelles (Le Fix 'Model Collapse')
df = processor.create_positional_features(df)

print("Shape:", df.shape)

## 2. Description des Colonnes et Statistiques

In [4]:
print("Colonnes disponibles :")
print(df.columns.tolist())

# Focus sur les nouvelles features positionnelles
new_features = [
    'dist_ema_20', 'dist_ema_50', 'dist_ema_200', 
    'rsi_norm', 'atr_ratio', 'macd_norm'
]

print("\nStatistiques Descriptives (Nouvelles Features) :")
df[new_features].describe()

## 3. Analyse des Outliers
Les features comme `dist_ema` doivent être centrées autour de 0 (sauf tendance forte) et avoir une distribution quasi-normale (ou Laplace). Les outliers représentent les 'Black Swans'.

In [5]:
plt.figure(figsize=(15, 6))
sns.boxplot(data=df[new_features])
plt.title("Distribution des Features Positionnelles (Avant Scaling)")
plt.grid(True)
plt.show()

## 4. Vérification après RobustScaler
Vérifions comment le `RobustScaler` traite ces données.

In [6]:
df_clean = df.dropna().reset_index(drop=True)
X_scaled, labels, cols = processor.scale_features(df_clean)

# Reconstitution en DataFrame pour visualisation
df_scaled = pd.DataFrame(X_scaled, columns=cols)

plt.figure(figsize=(15, 6))
sns.boxplot(data=df_scaled[['dist_ema_20', 'dist_ema_50', 'dist_ema_200', 'atr_ratio', 'macd_norm']])
plt.title("Distribution après RobustScaler")
plt.grid(True)
plt.show()

## 5. Finalisation et Export (NewdataFinal.csv)
Nous appliquons l'étiquetage (Labeling) et sauvegardons le dataset final complet pour le modèle.

In [7]:
# 1. Appliquer le Labeling (si ce n'est pas déjà fait dans df)
# Note: processor.labeler contient la logique de LABELING.py
print("Calcul des Labels (Q-Standard, VWAP, Gap)...")
df = processor.labeler.calculate_q_labels(df)

# 2. Nettoyage final (DropNA pour les labels et indicateurs)
df_final_clean = df.dropna().reset_index(drop=True)

# 3. Scaling final
X_final_scaled, y_final_labels, final_cols = processor.scale_features(df_final_clean)

# 4. Création du DataFrame final
df_export = pd.DataFrame(X_final_scaled, columns=final_cols)

# Ajouter la cible (Target)
if 'Label_Q_Standard' in df_final_clean.columns:
    df_export['Target'] = df_final_clean['Label_Q_Standard']
    print("Target 'Label_Q_Standard' ajoutée.")
else:
    print("Attention: Target non trouvée.")

# 5. Sauvegarde
output_csv_path = "../../../data/NewdataFinal.csv"
if not os.path.exists(os.path.dirname(output_csv_path)):
    # Fallback path if running with different root
    output_csv_path = "data/NewdataFinal.csv"
    
df_export.to_csv(output_csv_path, index=False)
print(f"\nDonnées prétraitées sauvegardées dans : {os.path.abspath(output_csv_path)}")
print(f"Dimensions: {df_export.shape}")
print("Prêt pour le Model Retraining !")

Calcul des Labels (Q-Standard, VWAP, Gap)...
Scaling features...
Selected features for training: ['dist_ema_20', 'dist_ema_50', 'dist_ema_200', 'rsi_norm', 'atr_ratio', 'macd_norm', 'macd_sig_norm', 'macd_hist_norm', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'Volume_log']
Saving scaler to artifacts/scaler.pkl...
Target 'Label_Q_Standard' ajoutée.

Données prétraitées sauvegardées dans : c:\Users\saady\Documents\project_trading_ai\Ai_Trading\data\NewdataFinal.csv
Dimensions: (280834, 16)
Prêt pour le Model Retraining !
